# 11 — Model Comparison & Final Selection
**Hotel Booking Demand Dataset**

This notebook loads all three trained models, builds a clean comparison table,
plots all ROC curves on a single chart, and selects the best model for the
Week 4 dashboard with full business justification.

| Model | Notebook |
|-------|---------|
| Logistic Regression | `08_logistic_regression.ipynb` |
| Decision Tree (depth=6) | `09_decision_tree.ipynb` |
| Random Forest (tuned) | `10_random_forest.ipynb` |


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import pickle, warnings, os

from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.dpi": 130, "font.size": 11})

MODEL_COLORS = {
    "Logistic Regression" : "#d62728",
    "Decision Tree"       : "#ff7f0e",
    "Random Forest"       : "#2ca02c",
}


In [ ]:
base = "../data/cleaned/"

with open(base + "X_test.pkl",         "rb") as f: X_test   = pickle.load(f)
with open(base + "y_test.pkl",         "rb") as f: y_test   = pickle.load(f)
with open(base + "model_lr.pkl",       "rb") as f: model_lr = pickle.load(f)
with open(base + "model_dt.pkl",       "rb") as f: model_dt = pickle.load(f)
with open(base + "model_rf_tuned.pkl", "rb") as f: model_rf = pickle.load(f)

MODELS = {
    "Logistic Regression": model_lr,
    "Decision Tree":       model_dt,
    "Random Forest":       model_rf,
}

print(f"Test set  : {X_test.shape[0]:,} samples | {X_test.shape[1]} features")
print(f"Class dist: {(y_test==0).sum():,} not-cancelled ({(y_test==0).mean()*100:.1f}%) "
      f"| {(y_test==1).sum():,} cancelled ({y_test.mean()*100:.1f}%)")


---
## 1 — Model Comparison Table

In [ ]:
rows = []
for name, model in MODELS.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    p, r, f, _ = precision_recall_fscore_support(y_test, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    rows.append({
        "Model"        : name,
        "Accuracy"     : round(accuracy_score(y_test, y_pred), 4),
        "Precision"    : round(p[1], 4),
        "Recall"       : round(r[1], 4),
        "F1-Score"     : round(f[1], 4),
        "ROC-AUC"      : round(roc_auc_score(y_test, y_prob), 4),
        "TP" : tp, "FP": fp, "FN": fn, "TN": tn,
    })

results_df = pd.DataFrame(rows).set_index("Model")

# Pretty display table
display_cols = ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]
print("=" * 72)
print("  MODEL COMPARISON — Cancelled Class (class 1) Metrics")
print("=" * 72)
print(results_df[display_cols].to_string())
print()
print("Best per metric:")
for col in display_cols:
    best = results_df[col].idxmax()
    print(f"  {col:<12}: {best:<25} ({results_df.loc[best, col]:.4f})")
print("=" * 72)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle("Model Comparison — Key Metrics & Revenue Impact",
             fontsize=13, fontweight="bold", y=1.02)

metric_cols = ["Accuracy", "Recall", "F1-Score", "ROC-AUC"]
model_names = results_df.index.tolist()
x = np.arange(len(metric_cols))
w = 0.25

for i, name in enumerate(model_names):
    vals   = [results_df.loc[name, c] for c in metric_cols]
    offset = (i - 1) * w
    bars   = axes[0].bar(x + offset, vals, w,
                         color=MODEL_COLORS[name], edgecolor="white",
                         alpha=0.9, label=name)
    for bar, val in zip(bars, vals):
        axes[0].text(bar.get_x() + bar.get_width()/2,
                     bar.get_height() + 0.008,
                     f"{val:.3f}", ha="center", fontsize=8, fontweight="bold")

axes[0].set_xticks(x)
axes[0].set_xticklabels(metric_cols, fontsize=10)
axes[0].set_ylabel("Score", fontsize=10)
axes[0].set_ylim(0, 1.12)
axes[0].set_title("Metric Comparison (class 1 = Cancelled)",
                  fontsize=11, fontweight="bold")
axes[0].legend(fontsize=10)

# Revenue-at-risk (FN × avg booking value scaled to full dataset)
AVG_BOOKING_VALUE = 107.51 * 3.75   # ADR × avg nights = £403
SCALE_FACTOR = 5                     # test is 20% of full data

fn_vals  = [int(results_df.loc[n, "FN"]) for n in model_names]
rev_risk = [fn * AVG_BOOKING_VALUE * SCALE_FACTOR for fn in fn_vals]

bars2 = axes[1].bar(model_names, rev_risk,
                    color=[MODEL_COLORS[n] for n in model_names],
                    edgecolor="white", width=0.5)
axes[1].set_title("Revenue at Risk from Missed Cancellations\n"
                  "(FN × £403 avg booking × 5× dataset scale)",
                  fontsize=11, fontweight="bold")
axes[1].set_ylabel("Estimated Revenue Unprotected (£)", fontsize=10)
axes[1].yaxis.set_major_formatter(
    mtick.FuncFormatter(lambda v, _: f"£{v/1e6:.2f}M"))

for bar, fn_n, rev in zip(bars2, fn_vals, rev_risk):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 15000,
                 f"{fn_n:,} FN\n£{rev/1e6:.2f}M",
                 ha="center", va="bottom", fontsize=10, fontweight="bold")
axes[1].set_ylim(0, max(rev_risk) * 1.28)

plt.tight_layout()
plt.savefig("../visuals/model_comparison_summary.png",
            dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print("Saved -> visuals/model_comparison_summary.png")


---
## 2 — Combined ROC Curves (All 3 Models)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("Model Comparison — ROC & Precision-Recall Curves",
             fontsize=13, fontweight="bold", y=1.02)

# ── ROC curves ────────────────────────────────────────────────────────────────
for name, model in MODELS.items():
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    is_best = (name == "Random Forest")

    axes[0].plot(fpr, tpr,
                 color=MODEL_COLORS[name],
                 lw=2.8 if is_best else 2.0,
                 linestyle="-" if is_best else "--",
                 label=f"{name}  (AUC = {auc:.4f})")
    if is_best:
        axes[0].fill_between(fpr, tpr, alpha=0.08, color=MODEL_COLORS[name])

axes[0].plot([0, 1], [0, 1], "k--", lw=1.2, alpha=0.4, label="Random  (AUC = 0.5000)")
axes[0].set_xlabel("False Positive Rate", fontsize=11)
axes[0].set_ylabel("True Positive Rate (Recall)", fontsize=11)
axes[0].set_title("ROC Curves — All Models", fontsize=11, fontweight="bold")
axes[0].legend(fontsize=10, loc="lower right")
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim([0, 1]); axes[0].set_ylim([0, 1.02])

# Add AUC improvement annotation
rf_auc = roc_auc_score(y_test, model_rf.predict_proba(X_test)[:, 1])
lr_auc = roc_auc_score(y_test, model_lr.predict_proba(X_test)[:, 1])
axes[0].annotate(f"RF leads by +{rf_auc - lr_auc:.4f} AUC\nover baseline LR",
                 xy=(0.4, 0.72), xytext=(0.52, 0.55),
                 fontsize=9, color="#2ca02c", fontweight="bold",
                 arrowprops=dict(arrowstyle="->", color="#2ca02c", lw=1.2))

# ── Precision-Recall curves ───────────────────────────────────────────────────
for name, model in MODELS.items():
    y_prob = model.predict_proba(X_test)[:, 1]
    prec, rec, _ = precision_recall_curve(y_test, y_prob)
    ap = average_precision_score(y_test, y_prob)
    is_best = (name == "Random Forest")

    axes[1].plot(rec, prec,
                 color=MODEL_COLORS[name],
                 lw=2.8 if is_best else 2.0,
                 linestyle="-" if is_best else "--",
                 label=f"{name}  (AP = {ap:.4f})")

baseline_pr = y_test.mean()
axes[1].axhline(baseline_pr, color="gray", linestyle=":", lw=1.2,
                label=f"Random baseline (AP = {baseline_pr:.2f})")
axes[1].set_xlabel("Recall", fontsize=11)
axes[1].set_ylabel("Precision", fontsize=11)
axes[1].set_title("Precision-Recall Curves (class 1 = Cancelled)",
                  fontsize=11, fontweight="bold")
axes[1].legend(fontsize=10, loc="upper right")
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim([0, 1]); axes[1].set_ylim([0, 1.05])

plt.tight_layout()
plt.savefig("../visuals/model_comparison_roc_pr.png",
            dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print("Saved -> visuals/model_comparison_roc_pr.png")


In [ ]:
# Save final selected model
final_path = "../data/cleaned/model_final.pkl"
with open(final_path, "wb") as f:
    pickle.dump(model_rf, f)

# Save comparison table to reports/
os.makedirs("../reports", exist_ok=True)
results_df.to_csv("../reports/model_comparison_results.csv")

print(f"Final model saved   : {final_path}")
print(f"Results CSV saved   : ../reports/model_comparison_results.csv")
print()
print("Final selected model: Random Forest (tuned)")
print(f"  ROC-AUC  : {results_df.loc['Random Forest', 'ROC-AUC']:.4f}")
print(f"  Recall   : {results_df.loc['Random Forest', 'Recall']:.4f}")
print(f"  F1-Score : {results_df.loc['Random Forest', 'F1-Score']:.4f}")


---
## 3 — Final Model Selection & Business Justification

### 3.1 Complete Comparison Table

| Model | Accuracy | Precision | Recall | F1-Score | **ROC-AUC** | False Negatives |
|-------|----------|-----------|--------|----------|------------|-----------------|
| Logistic Regression | 73.85% | 52.51% | 63.76% | 57.59% | 0.7852 | 1,728 |
| Decision Tree (depth=6) | 71.51% | 49.18% | 68.23% | 57.16% | 0.7862 | 1,515 |
| **Random Forest (tuned)** | **79.07%** | **62.65%** | 61.49% | **62.07%** | **0.8333** ✅ | 1,836 |

---

### 3.2 Which Model Is Best for This Business Case?

**Selected model: Random Forest (tuned)**

The Random Forest wins on four of the five key metrics:

| Criterion | Winner | Margin |
|-----------|--------|--------|
| ROC-AUC | **RF (0.8333)** | +0.0481 over LR, +0.0471 over DT |
| Accuracy | **RF (79.07%)** | +5.2 pp over LR, +7.6 pp over DT |
| F1-Score | **RF (62.07%)** | +4.5 pp over LR, +4.9 pp over DT |
| Precision | **RF (62.65%)** | Fewer false alarms — saves intervention cost |
| Recall | DT (68.23%) | DT catches 320 more cancellations |

The **Decision Tree has higher recall** (68.23% vs 61.49%) — it catches more
cancellations outright. However, the DT generates **1,614 more false positives**
than the RF (3,362 vs 1,748). At a conservative £10 cost per false-alarm
intervention, that's **£16,140 wasted per cycle** triggering retention actions
for guests who would have stayed regardless.

The Random Forest's **higher AUC (0.8333)** gives the decisive operational
advantage: the classification threshold can be **lowered from 0.5 to 0.35–0.40**
during peak season to recover high recall without permanently degrading precision
for lower-risk periods. A single fixed Decision Tree path cannot offer this
flexibility.

---

### 3.3 Why Recall Is More Important Than Accuracy

**Accuracy is misleading for this problem.**

A dummy model that predicts "not cancelled" for every booking achieves
**72.15% accuracy** — the exact class distribution. It would be completely
useless for revenue management while appearing to perform well.

The true cost structure is asymmetric:

| Error | What happens | Cost |
|-------|-------------|------|
| **False Negative** (missed cancellation) | Room sits empty. No overbooking buffer. No retention attempt. Revenue lost permanently. | **~£403** avg booking value — unrecoverable |
| **False Positive** (wrongly flagged) | Unnecessary re-confirmation call or upgrade offer. Guest stays anyway. | **~£5–15** intervention cost |

**Cost ratio: approximately 27:1 in favour of catching cancellations.**

Each of the 1,836 false negatives in the RF test set represents a booking where:
1. The hotel planned for an occupied room that stayed empty
2. No re-confirmation nudge was sent
3. The room was not released to last-minute channels
4. Full ADR × nights revenue was permanently lost

At scale across the full dataset:
```
1,836 FN × £403 avg value × 5× scale = ~£3.7M unprotected revenue
```

Recovering even 15% of these through earlier detection = **~£556K retained revenue**
— an enormous return on a modelling pipeline that costs nothing to run.

**ROC-AUC is the primary metric** because it evaluates the model across *all*
possible thresholds, not just 0.5. It tells us how well the model *ranks*
cancellation risk — which is exactly what a revenue manager needs to prioritise
their retention actions.

---

### 3.4 Selected Model for Week 4 Dashboard

**Final model: `model_final.pkl` (Random Forest, tuned)**

| Property | Value |
|----------|-------|
| Algorithm | `RandomForestClassifier` |
| Hyperparameters | `max_depth=20`, `min_samples_split=5`, `n_estimators=50`, `max_features='sqrt'` |
| Training data | SMOTE-balanced (98,814 samples, 50:50) |
| **Test ROC-AUC** | **0.8333** |
| Test Recall (cl.1) | 61.49% |
| Test F1 (cl.1) | 62.07% |
| Saved as | `data/cleaned/model_final.pkl` |

**Recommended production threshold: 0.40** (lower than default 0.50 to increase
recall for the cancelled class without excessive false positives).

The Week 4 dashboard will:
1. Load `model_final.pkl` and the feature pipeline
2. Accept new booking inputs (batch or single)
3. Return a **cancellation risk score (0–100%)** per booking
4. Flag bookings above the threshold for revenue management action
5. Show the **top contributing risk factors** per booking using feature importance

> **Week 3 complete.** All three models trained, evaluated, and compared.
> Next: `12_dashboard.ipynb` — Interactive Plotly / Tableau dashboard.
